In [1]:
pip install sentence-transformers

OSError: [WinError 740] 요청한 작업을 수행하려면 권한 상승이 필요합니다

# 재현성 평가

In [2]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# ==========================================
# 1. 모델 로드
# ==========================================
print("Loading SentenceTransformer model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded successfully!")

# ==========================================
# 2. 경로 및 환경 설정
# ==========================================
# GT 파일 경로
GT_PATH = r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\260128_Gemini-3-pro-hyper_parameter_test_채점결과\260130_GT.csv"

# 모델별 Answer 폴더 경로
MODELS_INFO = {
    "relevance-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_relevance_리샘플링",
    "ensemble-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_ensemble_리샘플링",
    "multiagent-rag": r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\csv\2048_50_9_multiagent_리샘플링"
}

# STS 결과를 저장할 최종 출력 폴더
OUTPUT_DIR = r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output\3. STS 결과(2048_50_9)"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

target_columns = [
    "Synthesis method",
    "Crystallization method",
    "Doping",
    "Coating",
    "Active material to Conductive additive to Binder ratio",
    "Electrolyte solvent",
    "Electrolyte solvent ratio",
    "Additive"
]

# ==========================================
# 3. 유틸리티 함수 및 GT 로드
# ==========================================
# 기존 코드의 오타 수정 (gt -> gt_df)
try:
    gt_df = pd.read_csv(GT_PATH, encoding="cp949")
except UnicodeDecodeError:
    gt_df = pd.read_csv(GT_PATH, encoding="euc-kr")

def calculate_semantic_similarity(text1, text2):
    """Sentence-BERT를 사용한 Semantic Textual Similarity (STS) 점수 계산"""
    if not text1 or not text2:
        return ""
    emb1 = model.encode(text1, convert_to_tensor=True)
    emb2 = model.encode(text2, convert_to_tensor=True)
    score = util.pytorch_cos_sim(emb1, emb2).item()
    return round(score, 4)

# ==========================================
# 4. 메인 실행 루프 (2개 모델 x 10회차)
# ==========================================
print("\n=== STS 평가 시작 ===")

for model_name, ans_dir in MODELS_INFO.items():
    for i in range(1, 11):  # 1부터 10까지
        print(f"Processing STS: {model_name} | Iteration: {i} ...")
        
        # 파일 경로 구성
        ans_filename = f"{model_name}_{i}_resampling.xlsx"
        ans_path = os.path.join(ans_dir, ans_filename)
        
        if not os.path.exists(ans_path):
            print(f"  -> Skipping: File not found ({ans_path})")
            continue

        # LLM Answer 로드 (엑셀 파일)
        llm_df = pd.read_excel(ans_path)

        # sample name의 교집합 추출
        common_samples = [s for s in gt_df["Sample"] if s in set(llm_df["Sample"])]
        
        evaluated_rows = []

       # 샘플별 비교 및 평가
        for sample in common_samples:
            gt_row = gt_df[gt_df["Sample"] == sample].iloc[0]
            llm_row = llm_df[llm_df["Sample"] == sample].iloc[0]
            
            result = {"Sample": sample}
            
            for col in target_columns:
                gt_val = str(gt_row.get(col, "")).strip()
                llm_val = str(llm_row.get(col, "")).strip()
                score = calculate_semantic_similarity(gt_val, llm_val) if gt_val and llm_val else ""
                
                result[f"GT_{col}"] = gt_val
                result[f"LLM_{col}"] = llm_val
                result[f"STS_{col}"] = score
            
            evaluated_rows.append(result)

        if not evaluated_rows:
            continue

        # 평가 DataFrame 생성 및 저장
        evaluation_df = pd.DataFrame(evaluated_rows)
        save_name = f"STS_{model_name}_{i}.xlsx"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        
        evaluation_df.to_excel(save_path, index=False)
        print(f"  -> Done! Saved to: {save_name}")

print("\n=== 모든 STS 평가 완료! ===")

Loading SentenceTransformer model...
Model loaded successfully!

=== STS 평가 시작 ===
Processing STS: relevance-rag | Iteration: 1 ...
  -> Done! Saved to: STS_relevance-rag_1.xlsx
Processing STS: relevance-rag | Iteration: 2 ...
  -> Done! Saved to: STS_relevance-rag_2.xlsx
Processing STS: relevance-rag | Iteration: 3 ...
  -> Done! Saved to: STS_relevance-rag_3.xlsx
Processing STS: relevance-rag | Iteration: 4 ...
  -> Done! Saved to: STS_relevance-rag_4.xlsx
Processing STS: relevance-rag | Iteration: 5 ...
  -> Done! Saved to: STS_relevance-rag_5.xlsx
Processing STS: relevance-rag | Iteration: 6 ...
  -> Done! Saved to: STS_relevance-rag_6.xlsx
Processing STS: relevance-rag | Iteration: 7 ...
  -> Done! Saved to: STS_relevance-rag_7.xlsx
Processing STS: relevance-rag | Iteration: 8 ...
  -> Done! Saved to: STS_relevance-rag_8.xlsx
Processing STS: relevance-rag | Iteration: 9 ...
  -> Done! Saved to: STS_relevance-rag_9.xlsx
Processing STS: relevance-rag | Iteration: 10 ...
  -> Done! S

# 하이퍼파라미터 평가

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 모델 로드
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

hyper_method_name = "relevance-rag(2048_100_9)_resampling"

# 1. GT.csv와 relevance-rag.csv를 input으로 받는다
gt = pd.read_csv("C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/260130_GT.csv", encoding="cp949")
llm_df = pd.read_csv(f"C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/리샘플링_CSV/{hyper_method_name}.csv", encoding="cp949") ## .drop(columns="Additional treatment")

# 2. 동일한 sample name끼리 평가 대상 열에 대해 비교한다
target_columns = [
    "Synthesis method",
    "Crystallization method",
    "Doping",
    "Coating",
    "Active material to Conductive additive to Binder ratio",
    "Electrolyte solvent",
    "Electrolyte solvent ratio",
    "Additive"
]

# sample name의 교집합 추출
common_samples = [s for s in gt_df["Sample"] if s in set(llm_df["Sample"])]

# 3. 평가 결과 저장용 리스트
evaluated_rows = []

# STS 계산 함수
def calculate_semantic_similarity(text1, text2):
    """Sentence-BERT를 사용한 Semantic Textual Similarity (STS) 점수 계산"""
    if not text1 or not text2:
        return ""
    emb1 = model.encode(text1, convert_to_tensor=True)
    emb2 = model.encode(text2, convert_to_tensor=True)
    score = util.pytorch_cos_sim(emb1, emb2).item()
    return round(score, 4)

# 샘플별 비교 및 평가
for sample in common_samples:
    gt_row = gt_df[gt_df["Sample"] == sample].iloc[0]
    llm_row = llm_df[llm_df["Sample"] == sample].iloc[0]
    
    result = {"Sample": sample}
    
    for col in target_columns:
        gt_val = str(gt_row.get(col, "")).strip()
        llm_val = str(llm_row.get(col, "")).strip()
        score = calculate_semantic_similarity(gt_val, llm_val) if gt_val and llm_val else ""
        
        result[f"GT_{col}"] = gt_val
        result[f"LLM_{col}"] = llm_val
        result[f"STS_{col}"] = score
    
    evaluated_rows.append(result)

# 평가 DataFrame 생성 및 저장
evaluation_df = pd.DataFrame(evaluated_rows)
evaluation_df.to_csv(f"C:/Users/LECS/Desktop/JSY/10. AIFFEL/VOLTAI-main/code/JSY/260128_Gemini-3-pro-hyper_parameter_test_채점결과/STS_채점결과_원본/STS_{hyper_method_name}.csv", index=False, encoding="utf-8-sig")


KeyboardInterrupt: 